# FLEO Module – Interactive Demo
Runs a single forward pass through the FLEO module (no GPU needed).
Shows orthogonality verification and fuzzy loss computation.

In [ ]:
import sys, torch
sys.path.insert(0, '..')
from models import FLEOModule
from models.loss import FuzzyConfusionLoss
from utils.confusion_matrix import FACS_PRIOR_7, normalize_confusion
from utils.metrics import orthogonality_check

## 1. Forward pass through FLEO

In [ ]:
B, C, H, W = 4, 256, 40, 40   # Typical P3 feature map
K, d = 7, 32

fleo = FLEOModule(in_channels=C, num_emotions=K, subspace_dim=d)
fleo.train()

x = torch.randn(B, C, H, W)
out, bind_logits = fleo(x)

print(f'Input  shape : {x.shape}')
print(f'Output shape : {out.shape}   (same as input – drop-in compatible)')
print(f'Bind logits  : {bind_logits.shape}  (B, K, K)')

## 2. Verify Gram-Schmidt guarantee: ⟨eᵢ, eⱼ⟩ = 0

In [ ]:
from models.fleo_module import GramSchmidtOrthogonalizer

gs = GramSchmidtOrthogonalizer()
v = torch.randn(B, K, d)       # random subspace vectors (NOT orthogonal)
e = gs(v)                       # after orthogonalization

score = orthogonality_check(e)
print(f'Mean |⟨eᵢ, eⱼ⟩| off-diagonal: {score:.6f}  (ideal = 0.0)')
assert score < 1e-5, 'Orthogonality not achieved!'
print('✓ Proposition 1 verified: all emotion subspaces are orthonormal.')

## 3. Fuzzy Confusion Loss

In [ ]:
C_prior = normalize_confusion(FACS_PRIOR_7)
print('Confusion Prior (normalized):')
print(C_prior.numpy().round(3))

fuzzy_loss = FuzzyConfusionLoss(C_prior)
logits = torch.randn(B, K)
targets = torch.randint(0, K, (B,))

loss_val = fuzzy_loss(logits, targets)
print(f'\nFuzzy loss value: {loss_val.item():.4f}')